In [4]:
import pandas as pd

def load_fin_2002(filepath):
    """
    Load 2002 Census of Governments Individual Unit file.
    Fixed-width ASCII, 35 chars per record, 1-indexed:
      cols 1-14:  Government Code
      cols 15-17: Item Code
      cols 18-29: Amount (in $1000)
      cols 30-35: Filler (blank)
    """
    colspecs = [(0, 14), (14, 17), (17, 29)]
    colnames = ['gov_code', 'item_code', 'amount']

    df = pd.read_fwf(
        filepath,
        colspecs=colspecs,
        names=colnames,
        header=None,
        dtype={'gov_code': str, 'item_code': str, 'amount': str}
    )

    # Parse gov_code subfields
    df['state_code'] = df['gov_code'].str[0:2]
    df['type_code']  = df['gov_code'].str[2:3]
    df['county_code']= df['gov_code'].str[3:6]
    df['unit_code']  = df['gov_code'].str[6:9]
    df['sup_code']   = df['gov_code'].str[9:12]
    df['sub_code']   = df['gov_code'].str[12:14]

    df['item_code']  = df['item_code'].str.strip()
    df['amount']     = pd.to_numeric(df['amount'].str.strip(), errors='coerce')

    return df


# --- Run ---
filepath = "/Users/mayarabin/Desktop/Thesis Downloads V2/Small Town:City Economic Data/2002_Individual Unit_file/2002FinInddiv24_NoImps.txt"

df_2002 = load_fin_2002(filepath)

print(df_2002.shape)
print(df_2002.head(10))
print(df_2002['item_code'].value_counts().head(20))

(1391960, 9)
         gov_code item_code   amount state_code type_code county_code  \
0  01000000000000       19H  2601431         01         0         000   
1  01000000000000       19T  1856481         01         0         000   
2  01000000000000       19X  1119175         01         0         000   
3  01000000000000       21G   203600         01         0         000   
4  01000000000000       21X   139955         01         0         000   
5  01000000000000       24G   281390         01         0         000   
6  01000000000000       24T   257417         01         0         000   
7  01000000000000       24X   329017         01         0         000   
8  01000000000000       31G    57480         01         0         000   
9  01000000000000       31X    44702         01         0         000   

  unit_code sup_code sub_code  
0       000      000       00  
1       000      000       00  
2       000      000       00  
3       000      000       00  
4       000      000   

In [5]:
def load_fingid_2002(filepath):
    """
    Load 2002 Census of Governments Individual Unit Directory File (02fingid.txt).
    Fixed-width ASCII, 160 chars per record, 1-indexed:
      cols 1-2:    State code
      cols 3-3:    Type code
      cols 4-6:    County code
      cols 7-9:    Unit code
      cols 10-12:  Sup code
      cols 13-14:  Sub code
      cols 15-78:  Government Name
      cols 79-113: County Name
      cols 114-115: FIPS State code
      cols 116-118: FIPS County code
      cols 119-123: FIPS Place code
      cols 124-132: Population
      cols 133-134: Population Year
      cols 135-141: Enrollment
      cols 142-143: Enrollment Year
      cols 144-145: Function code
      cols 146-147: School Level code
      cols 148-154: Weight
      cols 155-158: Fiscal Year End
      cols 159-160: Survey Year
    """
    colspecs = [
        (0, 2),    # state_code
        (2, 3),    # type_code
        (3, 6),    # county_code
        (6, 9),    # unit_code
        (9, 12),   # sup_code
        (12, 14),  # sub_code
        (14, 78),  # gov_name
        (78, 113), # county_name
        (113, 115),# fips_state
        (115, 118),# fips_county
        (118, 123),# fips_place
        (123, 132),# population
        (132, 134),# pop_year
        (134, 141),# enrollment
        (141, 143),# enrollment_year
        (143, 145),# function_code
        (145, 147),# school_level
        (147, 154),# weight
        (154, 158),# fiscal_year_end
        (158, 160),# survey_year
    ]
    colnames = [
        'state_code', 'type_code', 'county_code', 'unit_code', 'sup_code', 'sub_code',
        'gov_name', 'county_name', 'fips_state', 'fips_county', 'fips_place',
        'population', 'pop_year', 'enrollment', 'enrollment_year', 'function_code',
        'school_level', 'weight', 'fiscal_year_end', 'survey_year'
    ]

    df = pd.read_fwf(
        filepath,
        colspecs=colspecs,
        names=colnames,
        header=None,
        dtype=str
    )

    # Reconstruct gov_code to match the financial file
    df['gov_code'] = (
        df['state_code'] + df['type_code'] + df['county_code'] +
        df['unit_code'] + df['sup_code'] + df['sub_code']
    )

    df['gov_name']   = df['gov_name'].str.strip()
    df['county_name']= df['county_name'].str.strip()
    df['fips_place'] = df['fips_place'].str.strip().str.zfill(5)
    df['population'] = pd.to_numeric(df['population'], errors='coerce')

    return df


# --- Run ---
fingid_path = "/Users/mayarabin/Desktop/Thesis Downloads V2/Small Town:City Economic Data/2002_Individual Unit_file/02fingid.txt"

df_fingid = load_fingid_2002(fingid_path)

print(df_fingid.shape)
print(df_fingid.head(5)[['gov_code', 'gov_name', 'fips_state', 'fips_county', 'fips_place', 'type_code']])

(87575, 21)
         gov_code gov_name fips_state fips_county fips_place type_code
0  01000000000000  ALABAMA        NaN         NaN        NaN         0
1  01100100100000  AUTAUGA         01         001      99001         1
2  01100200200000  BALDWIN         01         003      99003         1
3  01100300300000  BARBOUR         01         005      99005         1
4  01100400400000     BIBB         01         007      99007         1


In [12]:
# Merge financial data with directory on gov_code
df_2002_merged = df_2002.merge(
    df_fingid[['gov_code', 'gov_name', 'county_name', 'fips_state', 'fips_county', 'fips_place', 'population']],
    on='gov_code',
    how='left'
)

print(df_2002_merged.shape)
print(df_2002_merged[['gov_code', 'gov_name', 'fips_place', 'item_code', 'amount', 'population', 'type_code']].head(10))

(1391960, 15)
         gov_code gov_name fips_place item_code   amount  population type_code
0  01000000000000  ALABAMA        NaN       19H  2601431   4447100.0         0
1  01000000000000  ALABAMA        NaN       19T  1856481   4447100.0         0
2  01000000000000  ALABAMA        NaN       19X  1119175   4447100.0         0
3  01000000000000  ALABAMA        NaN       21G   203600   4447100.0         0
4  01000000000000  ALABAMA        NaN       21X   139955   4447100.0         0
5  01000000000000  ALABAMA        NaN       24G   281390   4447100.0         0
6  01000000000000  ALABAMA        NaN       24T   257417   4447100.0         0
7  01000000000000  ALABAMA        NaN       24X   329017   4447100.0         0
8  01000000000000  ALABAMA        NaN       31G    57480   4447100.0         0
9  01000000000000  ALABAMA        NaN       31X    44702   4447100.0         0


In [15]:
df_2002_muni = df_2002_merged[df_2002_merged['type_code'].isin(['2', '3'])].copy()

b_codes = ['B01','B21','B22','B30','B42','B46','B50','B59','B79','B80','B89']
t_codes = ['T01','T09','T10','T11','T13','T16','T19','T20','T21','T22','T27','T28','T29']

target_codes = b_codes + t_codes

df_2002_filtered = df_2002_muni[df_2002_muni['item_code'].isin(target_codes)].copy()

# Separate B and T codes
df_b = df_2002_filtered[df_2002_filtered['item_code'].str.startswith('B')].groupby('gov_code')['amount'].sum().rename('total_transfers')
df_t = df_2002_filtered[df_2002_filtered['item_code'].str.startswith('T')].groupby('gov_code')['amount'].sum().rename('total_taxes')

# Collapse to one row per government with identifier columns
id_cols = ['gov_code', 'gov_name', 'county_name', 'fips_state', 'fips_county', 'fips_place', 'type_code', 'population']
df_id = df_2002_muni[id_cols].drop_duplicates(subset='gov_code')

# Merge together
df_2002_collapsed = df_id.merge(df_b, on='gov_code', how='left') \
                         .merge(df_t, on='gov_code', how='left')

print(df_2002_collapsed.shape)
print(df_2002_collapsed[['gov_code', 'gov_name', 'fips_place', 'total_transfers', 'total_taxes']].head(10))
print(df_2002_collapsed[['total_transfers', 'total_taxes']].describe())

(29283, 10)
         gov_code      gov_name fips_place  total_transfers  total_taxes
0  01200100100000  AUTAUGAVILLE      03220              NaN        119.0
1  01200200200000        DAPHNE      19648            285.0      10795.0
2  01200200400000      FAIRHOPE      25240            243.0       2367.0
3  01200200500000         FOLEY      26992            209.0       5547.0
4  01200200600000   ROBERTSDALE      65208              NaN       1385.0
5  01200220100000  SPANISH FORT      71976            214.0        367.0
6  01200250100000   GULF SHORES      32272            260.0       9266.0
7  01200250200000        LOXLEY      44608              NaN        800.0
8  01200300100000  BLUE SPRINGS      07672            500.0          NaN
9  01200300300000          CLIO      15640              NaN        161.0
       total_transfers   total_taxes
count     6.993000e+03  2.839700e+04
mean      2.073864e+03  3.458496e+03
std       4.759925e+04  8.073699e+04
min       0.000000e+00  0.000000e+00


In [16]:
# BEA Table 3.9.4, Line 1: Government consumption expenditures and gross investment
# 2002 index value = 67.610, base year 2017 = 100
BEA_2002 = 67.610
BEA_BASE = 100.0

deflator_2002 = BEA_BASE / BEA_2002  # ~1.4793

# Step 1: Deflate to 2017 real dollars (amounts are in $1000s nominal)
df_2002_collapsed['total_transfers_real'] = df_2002_collapsed['total_transfers'] * deflator_2002
df_2002_collapsed['total_taxes_real']     = df_2002_collapsed['total_taxes']     * deflator_2002

# Step 2: Convert from $1000s to dollars
df_2002_collapsed['total_transfers_real'] = df_2002_collapsed['total_transfers_real'] * 1000
df_2002_collapsed['total_taxes_real']     = df_2002_collapsed['total_taxes_real']     * 1000

# Step 3: Per capita
df_2002_collapsed['transfers_pc'] = df_2002_collapsed['total_transfers_real'] / df_2002_collapsed['population']
df_2002_collapsed['taxes_pc']     = df_2002_collapsed['total_taxes_real']     / df_2002_collapsed['population']

print(f"Deflator applied: {deflator_2002:.4f}")
print(df_2002_collapsed[['gov_name', 'population', 'total_transfers', 'transfers_pc', 'total_taxes', 'taxes_pc']].head(10))
print(df_2002_collapsed[['transfers_pc', 'taxes_pc']].describe())

Deflator applied: 1.4791
       gov_name  population  total_transfers  transfers_pc  total_taxes  \
0  AUTAUGAVILLE       820.0              NaN           NaN        119.0   
1        DAPHNE     16581.0            285.0     25.422790      10795.0   
2      FAIRHOPE     12480.0            243.0     28.799222       2367.0   
3         FOLEY      7590.0            209.0     40.728046       5547.0   
4   ROBERTSDALE      3782.0              NaN           NaN       1385.0   
5  SPANISH FORT      5423.0            214.0     58.366444        367.0   
6   GULF SHORES      5044.0            260.0     76.240781       9266.0   
7        LOXLEY      1348.0              NaN           NaN        800.0   
8  BLUE SPRINGS       121.0            500.0   6111.864229          NaN   
9          CLIO      2206.0              NaN           NaN        161.0   

      taxes_pc  
0   214.645690  
1   962.943911  
2   280.525753  
3  1080.949622  
4   541.648211  
5   100.095724  
6  2717.104126  
7   877.78702

In [19]:
PLACES_PATH = "/Users/mayarabin/Desktop/Thesis Files/places_panel_final.parquet"

# Load places panel
places = pd.read_parquet(PLACES_PATH)

In [21]:
import re

# ── Normalisation helpers ──────────────────────────────────────────────────
SUFFIXES = {'city', 'town', 'village', 'township', 'borough', 'municipality',
            'plantation', 'grant', 'location', 'purchase'}

def normalize_with_suffix(name):
    if not isinstance(name, str):
        return ''
    return re.sub(r'[^a-z0-9 ]', '', name.lower().strip())

def normalize_stripped(name):
    if not isinstance(name, str):
        return ''
    tokens = re.sub(r'[^a-z0-9 ]', '', name.lower().strip()).split()
    tokens = [t for t in tokens if t not in SUFFIXES]
    return ' '.join(tokens)

# ── MCD state set ──────────────────────────────────────────────────────────
MCD_STATES = {
    'Connecticut', 'Delaware', 'Indiana', 'Kansas', 'Maine', 'Maryland',
    'Massachusetts', 'Minnesota', 'New Hampshire', 'New Jersey', 'New York',
    'Pennsylvania', 'Rhode Island', 'Vermont', 'Virginia', 'Wisconsin'
}

# ── fips_state -> state_name mapping from places panel ────────────────────
fips_to_state = (places[['PLACE_ID', 'state_name']]
                 .assign(fips=places['PLACE_ID'].astype(str).str[:2])
                 .drop_duplicates(subset='fips')
                 .set_index('fips')['state_name']
                 .to_dict())

# ── Build PLACE_ID on financial data ──────────────────────────────────────
def build_geoid(row):
    state = fips_to_state.get(row['fips_state'], '')
    if state in MCD_STATES:
        return row['fips_state'] + row['fips_county'] + row['fips_place']
    else:
        return row['fips_state'] + row['fips_place']

df_2002_collapsed['fips_place'] = df_2002_collapsed['fips_place'].astype(str).str.strip().str.zfill(5)
df_2002_collapsed['fips_county'] = df_2002_collapsed['fips_county'].astype(str).str.strip().str.zfill(3)
df_2002_collapsed['fips_state'] = df_2002_collapsed['fips_state'].astype(str).str.strip().str.zfill(2)
df_2002_collapsed['PLACE_ID'] = df_2002_collapsed.apply(build_geoid, axis=1)

# ── Filter places panel to year 2002 ──────────────────────────────────────
places_2002 = places[places['year'] == 2000].copy()
places_2002['PLACE_ID'] = places_2002['PLACE_ID'].astype(str).str.strip()

print(f"Places panel 2002 slice: {len(places_2002):,} rows")
print(f"Financial PLACE_IDs — 7-digit: {(df_2002_collapsed['PLACE_ID'].str.len() == 7).sum():,}")
print(f"Financial PLACE_IDs — 10-digit: {(df_2002_collapsed['PLACE_ID'].str.len() == 10).sum():,}")

# ── Pass 1: direct PLACE_ID merge ─────────────────────────────────────────
merged_2002 = df_2002_collapsed.merge(places_2002, on='PLACE_ID', how='left')

print(f"\nAfter PLACE_ID merge — matched: {merged_2002['state_name'].notna().sum():,}, "
      f"unmatched: {merged_2002['state_name'].isna().sum():,}")

# ── Build name lookup tables from places_2002 ─────────────────────────────
lookup_name_with     = {}
lookup_name_stripped = {}

for _, row in places_2002.iterrows():
    place_id = str(row['PLACE_ID']).strip()
    state    = row['state_name']
    lookup_name_with.setdefault((state, normalize_with_suffix(row['NAME'])), []).append(place_id)
    lookup_name_stripped.setdefault((state, normalize_stripped(row['NAME'])), []).append(place_id)

# ── Pass 2: name matching on unmatched rows ───────────────────────────────
new_place_ids = {}
counts = {'matched_with': 0, 'matched_stripped': 0, 'unmatched': 0}

for idx in merged_2002[merged_2002['state_name'].isna()].index:
    fips     = merged_2002.at[idx, 'fips_state']
    state    = fips_to_state.get(fips, '')
    raw_name = merged_2002.at[idx, 'gov_name']

    if not state or not isinstance(raw_name, str):
        counts['unmatched'] += 1
        continue

    candidates = lookup_name_with.get((state, normalize_with_suffix(raw_name)), [])
    if len(candidates) == 1:
        new_place_ids[idx] = candidates[0]
        counts['matched_with'] += 1
        continue
    if len(candidates) > 1:
        counts['unmatched'] += 1
        continue

    candidates = lookup_name_stripped.get((state, normalize_stripped(raw_name)), [])
    if len(candidates) == 1:
        new_place_ids[idx] = candidates[0]
        counts['matched_stripped'] += 1
    else:
        counts['unmatched'] += 1

print(f"\nPass 2 — matched with suffix: {counts['matched_with']:,}, "
      f"matched stripped: {counts['matched_stripped']:,}, "
      f"unmatched: {counts['unmatched']:,}")

# ── Update PLACE_IDs and re-merge ─────────────────────────────────────────
for idx, place_id in new_place_ids.items():
    merged_2002.at[idx, 'PLACE_ID'] = place_id

places_cols = [c for c in places_2002.columns if c != 'PLACE_ID']
merged_2002 = merged_2002.drop(columns=places_cols).merge(places_2002, on='PLACE_ID', how='left')

print(f"\nFinal matched: {merged_2002['state_name'].notna().sum():,}, "
      f"unmatched: {merged_2002['state_name'].isna().sum():,}")

# ── Unmatched breakdown by state ──────────────────────────────────────────
unmatched_2002 = merged_2002[merged_2002['state_name'].isna()].copy()
unmatched_2002['state_name_derived'] = unmatched_2002['fips_state'].map(fips_to_state)
print(unmatched_2002['state_name_derived'].value_counts().head(25))

Places panel 2002 slice: 30,724 rows
Financial PLACE_IDs — 7-digit: 15,957
Financial PLACE_IDs — 10-digit: 13,326

After PLACE_ID merge — matched: 22,675, unmatched: 6,608

Pass 2 — matched with suffix: 777, matched stripped: 1,332, unmatched: 4,499

Final matched: 24,784, unmatched: 4,499
state_name_derived
Illinois        801
North Dakota    692
Ohio            470
Indiana         452
Kansas          356
New York        333
Nebraska        296
South Dakota    267
Michigan        144
Missouri        123
Virginia         93
Maryland         55
Delaware         24
Minnesota        15
Georgia          11
Vermont          11
Pennsylvania      9
Utah              9
Kentucky          9
Wisconsin         7
Oklahoma          6
Alabama           5
Arkansas          5
Maine             4
Texas             4
Name: count, dtype: int64
